# Zotero Client Examples

Demonstrates reading from a Zotero group library, exporting to BibTeX,
filtering by collection and tags, and writing back modified items.

**Prerequisites:**
- `pip install paperbridge[zotero,bibtex]`
- `.env` with `ZOTERO_API_KEY` and `ZOTERO_GROUP_ID` (or `ZOTERO_USER_ID` for personal libraries)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("ZOTERO_API_KEY")
USER_ID = os.getenv("ZOTERO_USER_ID")
GROUP_ID = os.getenv("ZOTERO_GROUP_ID")

In [2]:
from paperbridge.clients import ZoteroClient

# Pass group_id for a group library, or user_id for a personal library.
# If both are set, group_id takes precedence.
zot = ZoteroClient(api_key=API_KEY, user_id=USER_ID, group_id=GROUP_ID)
print(zot)

ZoteroClient(group_id='6437518')


## 1. Read entire group library and export to .bib

Fetch all items and call `item.to_bibtex()` on each one.
This preserves all Zotero fields — abstract, volume, issue, pages, ISSN, URL, tags, etc.

In [3]:
# Fetch every item in the library (auto-paginates)
all_items = zot.get_all_items()
print(f"Total items in library: {len(all_items)}")

# Preview first few
for item in all_items[:5]:
    print(f"  [{item.data.item_type}] {item.data.title} ({item.data.date})")

Total items in library: 337
  [journalArticle] EVA-MED: An Enhanced Valence-Arousal Multimodal Emotion Dataset for Emotion Recognition (2025)
  [preprint] A Review of Cognitive Readiness, Wearable Devices, and Prospects (2025)
  [attachment] Full Text PDF (None)
  [attachment] PubMed entry (None)
  [journalArticle] Psilocybin triggers an activity-dependent rewiring of large-scale cortical networks (2026-01-22)


In [4]:
from pathlib import Path

bib_entries = []
skipped = 0

for item in all_items:
    # Skip attachments and notes — they aren't citable items
    if item.data.item_type in ("attachment", "note"):
        skipped += 1
        continue

    bib_entries.append(item.to_bibtex())

print(f"Generated {len(bib_entries)} BibTeX entries (skipped {skipped} attachments/notes)")

Generated 272 BibTeX entries (skipped 65 attachments/notes)


In [5]:
# Write combined .bib file
output_path = Path("zotero_library.bib")
output_path.write_text("\n\n".join(bib_entries), encoding="utf-8")
print(f"Wrote {len(bib_entries)} entries to {output_path.resolve()}")

# Preview first entry
print("\n--- First entry ---")
print(bib_entries[0] if bib_entries else "(no entries)")

Wrote 272 entries to /Volumes/DataDrive/repo/nranthony/paperbridge/notebooks/zotero_library.bib

--- First entry ---
@article{unknown_evamed_2025,
	title = {EVA-MED: An Enhanced Valence-Arousal Multimodal Emotion Dataset for Emotion Recognition},
	url = {https://arxiv.org/abs/2503.16584},
	doi = {10.48550/ARXIV.2503.16584},
	abstract = {We introduce a novel multimodal emotion recognition dataset that enhances the precision of Valence-Arousal Model while accounting for individual differences. This dataset includes electroencephalography (EEG), electrocardiography (ECG), and pulse interval (PI) from 64 participants. Data collection employed two emotion induction paradigms: video stimuli that targeted different valence levels (positive, neutral, and negative) and the Mannheim Multicomponent Stress Test (MMST), which induced high arousal through cognitive, emotional, and social stressors. To enrich the dataset, participants' personality traits, anxiety, depression, and emotional states wer

## 2. Read the 'SENSE' collection and filter data-related tags

Find the collection by name, fetch its items, then extract and filter
tags that relate to data (datasets, databases, measurements, etc.).

In [6]:
# Find the 'SENSE' collection by name
collections = zot.get_collections(limit=100)
sense_col = next((c for c in collections if c.name == "SENSE"), None)

if sense_col is None:
    print("Collection 'SENSE' not found. Available collections:")
    for c in collections:
        print(f"  - {c.name} (key={c.key})")
else:
    print(f"Found collection: {sense_col.name} (key={sense_col.key})")

Found collection: SENSE (key=9TGS978Z)


In [7]:
# Fetch all items in the SENSE collection
if sense_col:
    sense_items = zot.get_items_in_collection(sense_col.key, limit=100)
    print(f"Items in SENSE: {sense_items.total_results}")

    for item in sense_items.items[:5]:
        tags_str = ", ".join(t.tag for t in item.data.tags)
        print(f"  {item.data.title[:60]}...  tags=[{tags_str}]")

Items in SENSE: 211
  A Review of Cognitive Readiness, Wearable Devices, and Prosp...  tags=[]
  RSA reactivity to parent-child conflict as a predictor of dy...  tags=[BioLab, ECG, Emotion Dysregulation, HRV, MindWare, Parent-Child Conflict, Psychophysiology, RSA, daily life, emotion dysregulation, parent-child conflict]
  Bringing the laboratory into the home: A protocol for remote...  tags=[Biobehavioral, ECG, Emotion Dysregulation, HRV, MindWare, Pregnancy, RSA, Remote Data Collection, emotion dysregulation, pregnant, protocol, remote data collection]
  Calmer: a robot for managing acute pain effectively in prete...  tags=[ECG, HRV, MindWare, NICU, Pain, Preterm Infant, pain, preterm infant, robot]
  From lab to life: Evaluating the reliability and validity of...  tags=[Ambulatory Assessment, ECG, EDA, HRV, MindWare, PPG, Psychophysiology, Reliability, Validity, Wearable, ambulatory assessment, reliability, validity, wearable]


In [8]:
# Collect all tags from the SENSE collection and filter for data-related ones
DATA_KEYWORDS = {
    "data", "dataset", "database", "sensor", "measurement",
    "accelerometer", "gyroscope", "imu", "wearable", "signal",
    "recording", "acquisition", "sampling", "time series",
}

if sense_col:
    all_tags = set()
    for item in sense_items.items:
        for t in item.data.tags:
            all_tags.add(t.tag)

    # Filter: tag matches if any data keyword appears in it (case-insensitive)
    data_tags = sorted(
        t for t in all_tags
        if any(kw in t.lower() for kw in DATA_KEYWORDS)
    )

    print(f"Total unique tags in SENSE: {len(all_tags)}")
    print(f"Data-related tags ({len(data_tags)}):")
    for tag in data_tags:
        print(f"  - {tag}")

Total unique tags in SENSE: 235
Data-related tags (4):
  - Remote Data Collection
  - Wearable
  - remote data collection
  - wearable


## 3. Modify and write a citation back to the group library

Pick an item, add a tag, and push the update back to Zotero.

In [9]:
# Pick the first item from the library as an example
result = zot.get_items(limit=1)
target = result.items[0]

print(f"Item: {target.data.title}")
print(f"Key:  {target.key}")
print(f"Current tags: {[t.tag for t in target.data.tags]}")

Item: EVA-MED: An Enhanced Valence-Arousal Multimodal Emotion Dataset for Emotion Recognition
Key:  S35DU9TH
Current tags: []


In [10]:
# Add a tag (this merges, so existing tags are preserved)
updated = zot.add_tags_to_item(target.key, ["paperbridge-reviewed"])

print(f"Updated tags: {[t.tag for t in updated.data.tags]}")

Updated tags: ['paperbridge-reviewed']


In [11]:
# You can also update other fields via update_item
from paperbridge.models.zotero import ZoteroItemData

# Example: update the Extra field to include a note
existing_extra = target.data.extra or ""
new_extra = f"{existing_extra}\nReviewed via paperbridge".strip()

updated = zot.update_item(
    target.key,
    ZoteroItemData(extra=new_extra),
)

print(f"Updated Extra field: {updated.data.extra}")

Updated Extra field: Reviewed via paperbridge


In [12]:
# Clean up the client when done
zot.close()